In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
pip install -U albumentations

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.9/269.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 632.7/632.7 kB 29.6 MB/s eta 0:00:00
  Attempting uninstall: albucore
    Found existing installation: albucore 0.0.19
    Uninstalling albucore-0.0.19:
      Successfully uninstalled albucore-0.0.19
  Attempting uninstall: albumentations
    Found existing installation: albumentations 1.4.20
    Uninstalling albumentations-1.4.20:
      Successfully uninstalled albumentations-1.4.20


In [3]:
!git clone https://github.com/ultralytics/ultralytics.git
!cd ultralytics && pip install .

Cloning into 'ultralytics'...
remote: Enumerating objects: 45745, done.
remote: Counting objects: 100% (648/648), done.
remote: Compressing objects: 100% (390/390), done.
remote: Total 45745 (delta 519), reused 266 (delta 258), pack-reused 45097 (from 5)
Receiving objects: 100% (45745/45745), 39.05 MiB | 14.87 MiB/s, done.
Resolving deltas: 100% (33869/33869), done.
Processing /content/ultralytics
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.52-py3-none-any.whl size=901706 sha256=ec18316231b2160b62e01512853c8f2aca03252f2cd800c1f708cb0ea838dd77
  Stored in directory: /tmp/pip-ephem-wheel-cache-l_16elnj/wheels/9a/cd/d5/95912172899f8ec640166ff6eef49156b1b00d6b2ade4a3cb1
Successfully built ultralytics


In [ ]:
import yaml
from ultralytics import YOLO
from google.colab import files, drive
import os

# Mount Google Drive to save weights
drive.mount('/content/drive')

# Paths to the unzipped dataset
train_images_path = '/content/drive/MyDrive/UAV/FOGGY/split_data/images/train'
val_images_path = '//content/drive/MyDrive/UAV/FOGGY/split_data/images/val'
train_labels_path = '/content/drive/MyDrive/UAV/FOGGY/split_data/labels/train'
val_labels_path = '/content/drive/MyDrive/UAV/FOGGY/split_data/labels/val'

# Create a temporary dictionary for data configuration
data_config = {
    'train': train_images_path,  # Path to training images
    'val': val_images_path,      # Path to validation images
    'nc': 35,                    # Number of classes
    'names': [                   # Class names
        "Traffic Signal", "Lamp Post", "Zebra Crossing", "Bike", "Car", "Rikshaw", "Tyre Works",
        "Tree", "Tractor", "Cattle", "Vegetation", "Electricity Pole", "Building", "Board", "Wall",
        "Person", "Bus", "Bridge", "Road Divider", "Tempo", "Traffic Sign Board", "Flag", "Crane",
        "Cycle", "Dog", "Truck", "Glove", "Overbridge", "Manhole", "Bus Stop", "Barricade",
        "Petrol Pump", "Ambulance", "Goat", "Cart"
    ]
}

# Save the configuration to a temporary file
config_file_path = '/content/drive/MyDrive/UAV/FOGGY/split_data (1).yaml'
with open(config_file_path, 'w') as file:
    yaml.dump(data_config, file)

print(f"Config file saved at {config_file_path}")

# Upload the weights file 'last.pt' to continue training
print("Please upload the weights file.")
uploaded = files.upload()  # Upload the file

# Print the names of the uploaded files for debugging
print("Uploaded files:", uploaded.keys())

# Ensure the uploaded file is correctly handled
uploaded_files = list(uploaded.keys())
if len(uploaded_files) == 0:
    raise FileNotFoundError("No file uploaded. Please upload the correct weights file.")
uploaded_file_name = uploaded_files[0]

# Handle file names like 'last (8).pt', 'last (9).pt', etc.
if 'last.pt' not in uploaded_file_name:
    # Rename the uploaded file to 'last.pt' if it contains 'last' and a number
    new_file_name = 'last.pt'
    os.rename(f'/content/{uploaded_file_name}', f'/content/{new_file_name}')
    uploaded_file_name = new_file_name
    print(f"Renamed the uploaded file to: {uploaded_file_name}")

# Path to the uploaded weights file
last_weights_path = f'/content/{uploaded_file_name}'
print(f"Using uploaded file: {uploaded_file_name}")

# Load the model from the uploaded weights
model = YOLO(last_weights_path)

# Resume training for 5 epochs or until 50 epochs total
total_epochs = 50
current_epoch = model.ckpt['epoch'] + 1  # Add 1 to the epoch because YOLOv8 starts from 0
remaining_epochs = total_epochs - current_epoch
epochs_to_train = min(35, remaining_epochs)  # Train for 5 or fewer epochs if near the target

if remaining_epochs > 0:
    print(f"Training for {epochs_to_train} epochs (Current Epoch: {current_epoch}).")
    model.train(data=config_file_path, epochs=current_epoch + epochs_to_train, batch=5)

    # Update the current epoch after training
    current_epoch += epochs_to_train

    # Save the updated weights
    new_weights_path = f'/content/drive/MyDrive/updated_weights_epoch_{current_epoch}.pt'
    model.save(new_weights_path)
    print(f"Updated weights saved at {new_weights_path}.")
else:
    print("Training is already complete. Total 50 epochs reached!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Config file saved at /content/drive/MyDrive/UAV/FOGGY/split_data (1).yaml
Please upload the weights file.


Saving last.pt to last.pt
Uploaded files: dict_keys(['last.pt'])
Using uploaded file: last.pt
Training for 35 epochs (Current Epoch: 0).
Ultralytics 8.3.52 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
engine/trainer: task=detect, mode=train, model=/content/last.pt, data=/content/drive/MyDrive/UAV/FOGGY/split_data (1).yaml, epochs=35, time=None, patience=100, batch=5, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, cla

100%|██████████| 755k/755k [00:00<00:00, 23.4MB/s]



                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128

100%|██████████| 5.35M/5.35M [00:00<00:00, 95.6MB/s]


AMP: checks passed ✅


In [ ]:
import shutil
from google.colab import files

# Specify the source directory and the target ZIP file path
source_dir = "/content/runs"
output_zip = "/content/runs.zip"

# Create a ZIP file from the source directory
shutil.make_archive(output_zip.replace('.zip', ''), 'zip', source_dir)

# Download the ZIP file to the local system
files.download(output_zip)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>